# Example: Data Auditing

**Before you start:** Open from `Notebooks/examples/` (or adjust `project_root` in the setup cell). Replace `path/to/images` and the metadata template with your data.

**Why audit?** Catching inconsistent image sizes, bad files, or image–mask mismatches early avoids failed or misleading runs later.

Full flow: `find_files` → `audit_data()` → interpret results → `process_images`. See [Common Workflows](../../docs/examples/workflows.md) and [Best Practices](../../docs/guides/best-practices.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

Imports the pipeline and adds the repository root so `phenome` resolves when you run the notebook from this folder.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from phenome import PhenoMe, get_metadata_from_path

## 2. Find Files

Use `find_files` with `mask_dir` when masks exist (for property presets). Use `get_metadata_from_path` for condition-based layouts.

In [ ]:
pheno = PhenoMe(seed=42)

image_dir = "path/to/images"  # e.g. data/images/
mask_dir = "path/to/masks"  # Optional; set to None if no masks

metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
file_df = pheno.find_files(
    image_dir,
    mask_dir=mask_dir if mask_dir and os.path.exists(mask_dir) else None,
    metadata_fn=metadata_fn,
    extensions=[".tif", ".tiff", ".png"],
)

print(f"Found {len(file_df)} images")
if "condition" in file_df.columns:
    print(f"Conditions: {file_df['condition'].unique().tolist()}")

## 3. Audit Data

Run `audit_data()` to get dimensions, intensity ranges, and image-mask shape matching.

In [ ]:
audit_df = pheno.audit_data()
print(audit_df)

## 4. Interpret and Process

Use audit results to choose `resize_size`, `pad_size`, and `channel_mode`. Then run the full pipeline.

In [ ]:
from phenome import load_dinov2_model

model, wrapper = load_dinov2_model()

# Process with parameters informed by audit
pheno.process_images(
    wrapper,
    channel_mode="split",  # Use 'split' for fluorescence
    resize_size=224,
)

# Optional: compute properties (requires masks for shape-based presets)
# pheno.compute_properties(property_preset='basic')

# Distance analysis and visualization
dist_results = pheno.compute_reference_distances(
    reference_filters={"condition": "Control"}, source="embeddings"
)
pheno.plot_pca(color_by="condition")
pheno.plot_distance_distribution(dist_results, group_by="condition")

## See Also

- [03_data_auditing_tutorial.ipynb](../tutorials/03_data_auditing_tutorial.ipynb): Step-by-step auditing tutorial (interpretation tips and troubleshooting)
- [04_exploratory_analysis.ipynb](04_exploratory_analysis.ipynb): Analysis after embeddings (clustering, outliers, summaries)
- [example_advanced_phenotyping.py](example_advanced_phenotyping.py): Full script-style pipeline
- [workflows.md](../../docs/examples/workflows.md): Basic Analysis Workflow